# Phase 1 — Master Tables

**Goal:** turn the raw Wyscout files into clean, validated foundation tables.
No ratings are computed here — this phase only produces trustworthy inputs.

Outputs (written to `data/`):

| Table | Grain | Purpose |
|---|---|---|
| `players_master` | one row / player | biographical + GK/DF/MD/FW position |
| `teams_master` | one row / team | club & national team names |
| `matches_master` | one row / match | scores, coaches, ET/penalty flags |
| `player_appearances` | one row / (player, match) | **minutes played** + authoritative goals/assists/cards |
| `events_enriched/<comp>.parquet` | one row / event | every event flagged (accurate, goal, progressive…) |

The single most important number produced here is **minutes_played** — it is
the denominator of every per-90 metric downstream. We validate it explicitly.

In [1]:
import sys, time
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))   # make wyscout_lib importable
import wyscout_lib as wl

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
DATA = wl.DATA
print("Competitions:", ", ".join(wl.COMPETITIONS))

Competitions: england, france, germany, italy, spain, euro_2016, world_cup


## 1. Players & Teams
Straight reads of the biographical files. `position_code` drives every
position-normalised percentile later, so we confirm its distribution.

In [2]:
players = wl.load_players_master()
teams = wl.load_teams_master()
players.to_parquet(DATA / "players_master.parquet")
teams.to_parquet(DATA / "teams_master.parquet")

print(f"players_master: {players.shape}")
print(players["position_code"].value_counts().to_string())
players.head(3)

players_master: (3603, 11)
position_code
MD    1257
DF    1200
FW     720
GK     426


,player_id,first_name,last_name,short_name,birth_date,nationality,position_code,height_cm,weight_kg,preferred_foot,current_team_id
0,32777,Harun,Tekin,H. Tekin,1989-06-17,Turkey,GK,187,78,right,4502
1,393228,Malang,Sarr,M. Sarr,1999-01-23,Senegal,DF,182,73,left,3775
2,393230,Over,Mandanda,O. Mandanda,1998-10-26,France,GK,176,72,NaN,3772


## 2. Matches & Appearances

We loop over all 7 competitions. For every match we record scores, the two
coaches, and whether it went to extra time / penalties. For every player in a
lineup or on the bench we compute **minutes_played** and carry the
authoritative `goals` / `assists` / `own_goals` / cards straight from the
match sheet (these are exact — they are not inferred from event tags).

In [3]:
t = time.time()
all_matches, all_apps = [], []
for comp in wl.COMPETITIONS:
    mrows, arows = wl.build_matches_and_appearances(comp)
    all_matches.extend(mrows)
    all_apps.extend(arows)
    print(f"  {comp:14s} matches={len(mrows):4d}  appearances={len(arows):5d}")

matches = pd.DataFrame(all_matches)
appearances = pd.DataFrame(all_apps)
matches.to_parquet(DATA / "matches_master.parquet")
appearances.to_parquet(DATA / "player_appearances.parquet")
print(f"\nmatches_master: {matches.shape}  |  player_appearances: {appearances.shape}  ({time.time()-t:.1f}s)")

  england        matches= 380  appearances=13668
  france         matches= 380  appearances=13664
  germany        matches= 306  appearances=11002
  italy          matches= 380  appearances=16880
  spain          matches= 380  appearances=13673
  euro_2016      matches=  51  appearances= 2326
  world_cup      matches=  64  appearances= 2885



matches_master: (1941, 16)  |  player_appearances: (74098, 11)  (0.2s)


## 3. Minutes validation
Hard checks — if any of these fail, every downstream per-90 is suspect.

In [4]:
played = appearances[appearances.minutes_played > 0]
assert appearances.minutes_played.between(0, 120).all(), "minutes outside [0,120]!"
print("✓ all minutes in [0, 120]")
print(f"✓ appearances with minutes > 0: {len(played):,}")
print(f"✓ unused substitutes (0 min): {(appearances.minutes_played==0).sum():,}")
print(f"  extra-time matches: {matches.has_extra_time.sum()}  |  penalty shootouts: {matches.has_penalties.sum()}")

# total player-minutes per competition vs naive expectation (matches × 90 × ~21.9 players)
chk = (played.merge(matches[["match_id","competition_id"]].drop_duplicates(), on=["match_id"], suffixes=("","_m"))
            .groupby("competition_id").minutes_played.agg(["sum","count"]))
chk["matches"] = matches.groupby("competition_id").size()
chk["minutes_per_match"] = chk["sum"] / chk["matches"]
print("\nplayer-minutes per match (expect ~1980 = 22 players × 90):")
print(chk[["matches","minutes_per_match"]].round(0).to_string())

✓ all minutes in [0, 120]
✓ appearances with minutes > 0: 53,380
✓ unused substitutes (0 min): 20,718
  extra-time matches: 10  |  penalty shootouts: 7

player-minutes per match (expect ~1980 = 22 players × 90):
                matches  minutes_per_match
competition_id                            
england             380             1977.0
euro_2016            51             2043.0
france              380             1974.0
germany             306             1976.0
italy               380             1973.0
spain               380             1976.0
world_cup            64             2029.0


## 4. Event enrichment

We flatten + flag every event, one competition at a time (memory-friendly),
and persist one parquet per competition under `data/events_enriched/`.
Enrichment adds the boolean columns the aggregator needs: `pass_accurate`,
`is_progressive_pass`, `event_goal`, `is_clearance`, etc.

In [5]:
ENR = DATA / "events_enriched"
ENR.mkdir(exist_ok=True)
t = time.time()
total = 0
enrich_summary = []
for comp in wl.COMPETITIONS:
    raw = wl.load_raw_events(comp)
    ev = wl.enrich_events(raw)
    ev.to_parquet(ENR / f"{comp}.parquet")
    total += len(ev)
    enrich_summary.append({"competition": comp, "events": len(ev),
                           "goals(event)": int(ev.event_goal.sum()),
                           "shots": int(ev.is_shot.sum()),
                           "passes": int(ev.is_pass.sum())})
    print(f"  {comp:14s} {len(ev):>8,} events  ({time.time()-t:5.1f}s elapsed)")
    del raw, ev

print(f"\nTOTAL enriched events: {total:,}")
pd.DataFrame(enrich_summary)

  england         643,150 events  (  6.0s elapsed)


  france          632,807 events  ( 11.8s elapsed)


  germany         519,407 events  ( 16.7s elapsed)


  italy           647,372 events  ( 22.5s elapsed)


  spain           628,659 events  ( 28.4s elapsed)


  euro_2016        78,140 events  ( 29.0s elapsed)


  world_cup       101,759 events  ( 29.7s elapsed)

TOTAL enriched events: 3,251,294


,competition,events,goals(event),shots,passes
0,england,643150,988,8451,328657
1,france,632807,998,8327,319198
2,germany,519407,833,6898,261462
3,italy,647372,980,8806,337317
4,spain,628659,994,7979,318722
5,euro_2016,78140,105,1198,43695
6,world_cup,101759,157,1419,56457


## Phase 1 complete
`players_master`, `teams_master`, `matches_master`, `player_appearances`, and
`events_enriched/*` are written to `data/`. Next: **Phase 2 — player_match_stats**.